Needs
1) if data needs some preprocessing 
2) at times data importing is a headache 
3) What if we need to shuffle multiple datasets 
4) How about extracting multiple batches and making them work paralelly

For all this we need 2 classes 
DATASET AND DATALOADER 


Dataset and dataloader are core abstractions in pytorch that decouple how you define your data from how you efficiently iterate over it in training loops.

Dataset - loads class 
Dataloader - makes batches
Dataset Class
This class is a blueprint. It decides how data is loaded and returned
It defines 
__init__() - how the data is loaded , from where it is loaded etc
__len__() - defines no. of samples in my dataset
__get_item__(index) - returns the item in my dataset at the index specified

### FLOW ###

The DATALOADER wraps the dataset and handles batching , shuffling and parallel loading

1) At start of each cycle the dataloader (if shuffle = True) shuffles indices(using a sampler)
2) It divides the indices into chunks of batch_size
3) then we send the custom indices to the __get_item(index)__ this would return the data from the dataset
4) then these samples are collected and combined into batch (using collate_fn)
5) then we send this to main training loop


In [1]:
from sklearn.datasets import make_classification
import torch

In [7]:
# create synthetic classification dataset using sklearn
X,y = make_classification(
    n_samples=10,
    n_features=2,
    n_informative=2,
    n_redundant=0,
    n_classes=2,
    random_state=42 
)

In [8]:
X

array([[ 1.06833894, -0.97007347],
       [-1.14021544, -0.83879234],
       [-2.8953973 ,  1.97686236],
       [-0.72063436, -0.96059253],
       [-1.96287438, -0.99225135],
       [-0.9382051 , -0.54304815],
       [ 1.72725924, -1.18582677],
       [ 1.77736657,  1.51157598],
       [ 1.89969252,  0.83444483],
       [-0.58723065, -1.97171753]])

In [9]:
y

array([1, 0, 0, 0, 0, 1, 1, 1, 1, 0])

In [10]:
from torch.utils.data import Dataset,DataLoader

In [ ]:
# create custom dataset class
class CustomDatasetClass(Dataset):
    def __init__(self,features,labels):
        self.features = features
        self.labels = labels
    
    def __len__(self):
        return self.features.shape[0]
    # can do any transformation in this getitem function too
    def __getitem__(self,index):
        return self.features[index],self.labels[index]

In [22]:
Data = CustomDatasetClass(X,y)


In [24]:
dataloader = DataLoader(Data,batch_size=2,shuffle=True)

In [25]:
for batch_features,batch_labels in dataloader:
    print(batch_features)
    print(batch_labels)
    print("-"*50)

tensor([[-0.5872, -1.9717],
        [ 1.0683, -0.9701]], dtype=torch.float64)
tensor([0, 1])
--------------------------------------------------
tensor([[-2.8954,  1.9769],
        [-1.1402, -0.8388]], dtype=torch.float64)
tensor([0, 0])
--------------------------------------------------
tensor([[-0.9382, -0.5430],
        [ 1.8997,  0.8344]], dtype=torch.float64)
tensor([1, 1])
--------------------------------------------------
tensor([[-1.9629, -0.9923],
        [-0.7206, -0.9606]], dtype=torch.float64)
tensor([0, 0])
--------------------------------------------------
tensor([[ 1.7774,  1.5116],
        [ 1.7273, -1.1858]], dtype=torch.float64)
tensor([1, 1])
--------------------------------------------------


### Worker

can do parellel operations side by side that means
processing of simultaneous batches makes everything faster 

### Sampler
derternmines the stratergy for selecting samnmples from the dataset during data loading, determines how the indices are drawin for each batch

1) Sequential Sampler - in the order the entries appear in the dataset (mostly used in time series data)
2) Random sampler - shuffle = True 
3) Custom sampler - make the Sampler work in my own way (if i want specific ratios of data)

### Collate-Fn
Specifies how to combine teh list of samples from the dataset into a single batch. The dataiader uses sinoke batch collation, we can customise how the data is collated.

### Imp Params
1) dataset - main funciton 
2) batch_size - how many sampler per batch to load
3) shuffle - shuffle dataset in each epoch 
4) num_workers - how many workers we need for parallelisation
5) pin_memory - Dataloader will copy tensors into piinnned(page-locked) memory before returning them
6) drop_last - drops the last incomplete batch if total samples is not divisible by batch_size
7) collat_fn - processes list of samples into a batch
8) sampler - defines start fro drawing samples